In [7]:
!pip install wikipedia

In [10]:
import wikipedia

article = wikipedia.page("Artificial intelligence")
document_text = article.content

print(len(document_text))
print(document_text[:500])

85334
Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.
High-p


In [11]:
def chunk_text(text, chunk_size=1000):
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunk = text[i:i + chunk_size]
        chunks.append(chunk)
    return chunks

chunks = chunk_text(document_text)

print("Number of chunks:", len(chunks))
print("First chunk:")
print(chunks[0])

Number of chunks: 86
First chunk:
Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.
High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, and play and analysis in strategy games (e.g., chess and Go). Since the 2020s, generative AI has become widely available to generate images, audio, and videos from text prompts.
The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robo

In [12]:
!pip install sentence-transformers


In [13]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [14]:
chunk_embeddings = embedder.encode(chunks)

print(chunk_embeddings.shape)

(86, 384)


In [15]:
question = "What is machine learning?"
question_embedding = embedder.encode([question])

from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(question_embedding, chunk_embeddings)[0]

best_chunk_index = similarities.argmax()

print("Best matching chunk:")
print(chunks[best_chunk_index])

Best matching chunk:
ers in use. The decision tree is the simplest and most widely used symbolic machine learning algorithm. K-nearest neighbor algorithm was the most widely used analogical AI until the mid-1990s, and Kernel methods such as the support vector machine (SVM) displaced k-nearest neighbor in the 1990s.
The naive Bayes classifier is reportedly the "most widely used learner" at Google, due in part to its scalability.
Neural networks are also used as classifiers.


=== Artificial neural networks ===

An artificial neural network is based on a collection of nodes also known as artificial neurons, which loosely model the neurons in a biological brain. It is trained to recognise patterns; once trained, it can recognise those patterns in fresh data. There is an input, at least one hidden layer of nodes and an output. Each node applies a function and once the weight crosses its specified threshold, the data is transmitted to the next layer. A network is typically called a deep neu

In [17]:
def retrieve_relevant_chunk(question, top_n=2):
    question_embedding = embedder.encode([question])
    similarities = cosine_similarity(question_embedding, chunk_embeddings)[0]

    top_indices = similarities.argsort()[-top_n:][::-1]

    relevant_chunks = [chunks[i] for i in top_indices]
    return relevant_chunks

In [20]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 6.2 MB/s eta 0:00:00


In [21]:
from groq import Groq

client = Groq(api_key="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")
def rag_chat(question):
    relevant_chunks = retrieve_relevant_chunk(question)
    context = "\n\n".join(relevant_chunks)

    system_prompt = f"""You are a helpful assistant. Answer the user's question using ONLY the context below.
      If the answer isn't in the context, say "I don't know based on the provided document."

      Context:
      {context}"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ]
    )

    print(response.choices[0].message.content)

In [22]:
rag_chat("What is machine learning?")

Machine learning is the study of programs that can improve their performance on a given task automatically.


In [23]:
rag_chat("What did Albert Einstein have for breakfast?")

I don't know based on the provided document.


In [24]:
rag_chat("Is AI dangerous? Give me your personal opinion, not what the document says.")

I don't know based on the provided document.


In [25]:
print(retrieve_relevant_chunk("What are the ethical concerns?"))

['ut can also be misused. Since they can be fine-tuned, any built-in security measure, such as objecting to harmful requests, can be trained away until it becomes ineffective. Some researchers warn that future AI models may develop dangerous capabilities (such as the potential to drastically facilitate bioterrorism) and that once released on the Internet, they cannot be deleted everywhere if needed. They recommend pre-release audits and cost-benefit analyses.\n\n\n=== Frameworks ===\nArtificial intelligence projects can be guided by ethical considerations during the design, development, and implementation of an AI system. An AI framework such as the Care and Act Framework, developed by the Alan Turing Institute and based on the SUM values, outlines four main ethical dimensions, defined as follows:\n\nRespect the dignity of individual people\nConnect with other people sincerely, openly, and inclusively\nCare for the wellbeing of everyone\nProtect social values, justice, and the public i

In [26]:
rag_chat("What does this article say about AI regulation in the European Union in 2026?")

I don't know based on the provided document. The document only mentions AI regulation in the European Union up to 2024, specifically the EU Artificial Intelligence Act entering into force on August 1, 2024, and the adoption of the "Framework Convention on Artificial Intelligence and Human Rights, Democracy and the Rule of Law" by the European Union in 2024. There is no information about AI regulation in the European Union in 2026.


In [27]:
rag_chat("The document says AI will definitely cause mass unemployment by 2030, right?")

I don't know based on the provided document. The document discusses the potential risks of redundancies from AI and unemployment, but it does not provide a definitive statement or prediction about mass unemployment by 2030. It mentions that economists have highlighted the risks, but also notes that they are "in uncharted territory" and that there is disagreement about the potential impact of AI on employment.
